# Code related to annotating with NDP View 2 
## Visualize Clustered cell distribution

In [1]:
import numpy as np
from torch.utils.data import DataLoader
import pyvips
from pathlib import Path
import random
import matplotlib.pyplot as plt

random.seed(25)
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from skan import draw, Skeleton
import pyvips
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from scripts.filters import *
from scripts.utils import *
import os
import xml.etree.ElementTree as ET
import xml.dom.minidom
import openslide

CLUSTER_COLORS = {
    1: "#FF0000",
    0: "#00CC00",
    2: "#0066FF",
    3: "#FF9900",
    5: "#CC00CC",
    4: "#00CCCC",
}


def write_ndpa(df, NDPI_PATH, slice=1, slide=None):
    filtered_df = df[(df["brain_slice"] == slice) & (df["slide"] == slide)].copy()

    filtered_df["label"] = ""
    filtered_df["color"] = (
        filtered_df["gmm_cluster"].map(CLUSTER_COLORS).fillna("#FF0000")
    )
    filtered_df["width"] = filtered_df["x2"] - filtered_df["x1"]
    filtered_df["height"] = filtered_df["y2"] - filtered_df["y1"]

    filtered_df = filtered_df.rename(columns={"x1": "x", "y1": "y"})

    BOUNDING_BOXES = filtered_df[
        ["label", "color", "x", "y", "width", "height"]
    ].to_dict(orient="records")

    def px_to_ndpa(px, py, img_w, img_h, mpp_x_nm, mpp_y_nm, offset_x_nm, offset_y_nm):
        """Convert a single pixel coordinate (top-left origin) to NDPA nanometres."""
        ndpa_x = (px - img_w / 2) * mpp_x_nm + offset_x_nm
        ndpa_y = (py - img_h / 2) * mpp_y_nm + offset_y_nm
        return int(round(ndpa_x)), int(round(ndpa_y))

    def build_ndpa(boxes, img_w, img_h, mpp_x_nm, mpp_y_nm, offset_x_nm, offset_y_nm):

        fragments = []

        for i, box in enumerate(boxes):
            label = box.get("label", f"bbox_{i + 1}")
            color = box.get("color", "#FF0000")

            x, y, w, h = box["x"], box["y"], box["width"], box["height"]

            cx_px = x + w / 2
            cy_px = y + h / 2
            cx_nm, cy_nm = px_to_ndpa(
                cx_px, cy_px, img_w, img_h, mpp_x_nm, mpp_y_nm, offset_x_nm, offset_y_nm
            )

            corners_px = [
                (x, y),
                (x + w, y),
                (x + w, y + h),
                (x, y + h),
            ]
            corners_nm = [
                px_to_ndpa(
                    cx2, cy2, img_w, img_h, mpp_x_nm, mpp_y_nm, offset_x_nm, offset_y_nm
                )
                for cx2, cy2 in corners_px
            ]

            state = ET.Element("ndpviewstate", id=str(i + 1))
            ET.SubElement(state, "title").text = label
            ET.SubElement(state, "details").text = ""
            ET.SubElement(state, "x").text = str(cx_nm)
            ET.SubElement(state, "y").text = str(cy_nm)
            ET.SubElement(state, "z").text = "0"
            ET.SubElement(state, "lens").text = "20.000000"

            ann = ET.SubElement(
                state, "annotation", type="freehand", displayname=label, color=color
            )
            ET.SubElement(ann, "measuretype").text = "0"
            ET.SubElement(ann, "closed").text = "1"
            ET.SubElement(ann, "specialtype").text = "rectangle"

            pl = ET.SubElement(ann, "pointlist")
            for nx, ny in corners_nm:
                pt = ET.SubElement(pl, "point")
                ET.SubElement(pt, "x").text = str(nx)
                ET.SubElement(pt, "y").text = str(ny)

            ET.SubElement(ann, "specialtype").text = "rectangle"

            fragments.append(ET.tostring(state, encoding="unicode"))

        return fragments

    def pretty_join(fragments):
        parts = []
        for frag in fragments:
            pretty = xml.dom.minidom.parseString(frag).toprettyxml(indent="  ")
            lines = pretty.splitlines()
            body = "\n".join(l for l in lines if not l.startswith("<?xml"))
            parts.append(body.strip())

        inner = "\n".join(parts)
        return (
            '<?xml version="1.0" encoding="utf-8" standalone="yes"?>\n'
            "<annotations>\n"
            f"{inner}\n"
            "</annotations>\n"
        )

    if not os.path.isfile(NDPI_PATH):
        raise FileNotFoundError(f"NDPI file not found: {NDPI_PATH}")

    print(f"Opening slide: {NDPI_PATH}")
    slide = openslide.OpenSlide(NDPI_PATH)
    props = slide.properties

    img_w, img_h = slide.dimensions
    print(f"  Dimensions: {img_w} x {img_h} px")

    mpp_x = float(props.get(openslide.PROPERTY_NAME_MPP_X, 0))
    mpp_y = float(props.get(openslide.PROPERTY_NAME_MPP_Y, 0))

    mpp_x_nm = mpp_x * 1000
    mpp_y_nm = mpp_y * 1000
    print(f"  Resolution: {mpp_x:.4f} µm/px  ({mpp_x_nm:.1f} nm/px)")

    offset_x_nm = float(props.get("hamamatsu.XOffsetFromSlideCentre", 0))
    offset_y_nm = float(props.get("hamamatsu.YOffsetFromSlideCentre", 0))

    slide.close()

    print(f"BBox count: {len(BOUNDING_BOXES)}")
    fragments = build_ndpa(
        BOUNDING_BOXES, img_w, img_h, mpp_x_nm, mpp_y_nm, offset_x_nm, offset_y_nm
    )

    out_path = NDPI_PATH + ".ndpa"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(pretty_join(fragments))

In [2]:
df = pd.read_parquet("./results/spatial/CTX_gmm.parquet")
vals = df["slide"].unique()
NDPI_DIR = "/mnt/d/microglia_data/"
ndpi_pathes = [f for f in Path(NDPI_DIR).glob("./*.ndpi")]
print(vals)
print(ndpi_pathes)

['TPO_60' 'TPO_61_EV' 'TPO_62_EV2' 'TPO_62_EV' 'TPO_63_TPO' 'TPO_64_EV2'
 'TPO_64_EV' 'TPO_65_TPO2' 'TPO_65_TPO' 'TPO_66_EV2' 'TPO_66_EV']
[PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_60 TPO_Iba1_Brain - 2026-04-21 16.45.55.ndpi'), PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_61 EV_Iba1_Brain - 2026-04-21 16.53.13.ndpi'), PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_62 EV_2_Iba1_Brain - 2026-04-21 17.07.21.ndpi'), PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_62 EV_Iba1_Brain - 2026-04-21 17.00.49.ndpi'), PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_63 TPO_Iba1_Brain - 2026-04-21 17.14.21.ndpi'), PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_64 EV_2_Iba1_Brain - 2026-04-21 17.28.33.ndpi'), PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_64 EV_Iba1_Brain - 2026-04-21 17.21.11.ndpi'), PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_65 TPO_2_Iba1_Brain - 2026-04-21 17.43.26.ndpi'), PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_65 TPO_Iba1_Brai

In [3]:
slice = 3
for i, path in enumerate(ndpi_pathes):
    write_ndpa(df, str(path), slice, vals[i])

Opening slide: /mnt/d/microglia_data/2502 Busulfan TPO_60 TPO_Iba1_Brain - 2026-04-21 16.45.55.ndpi
  Dimensions: 163840 x 39424 px
  Resolution: 0.2276 µm/px  (227.6 nm/px)
BBox count: 1008
Opening slide: /mnt/d/microglia_data/2502 Busulfan TPO_61 EV_Iba1_Brain - 2026-04-21 16.53.13.ndpi
  Dimensions: 167936 x 45824 px
  Resolution: 0.2276 µm/px  (227.6 nm/px)
BBox count: 1377
Opening slide: /mnt/d/microglia_data/2502 Busulfan TPO_62 EV_2_Iba1_Brain - 2026-04-21 17.07.21.ndpi
  Dimensions: 180224 x 44544 px
  Resolution: 0.2276 µm/px  (227.6 nm/px)
BBox count: 0
Opening slide: /mnt/d/microglia_data/2502 Busulfan TPO_62 EV_Iba1_Brain - 2026-04-21 17.00.49.ndpi
  Dimensions: 143360 x 37120 px
  Resolution: 0.2276 µm/px  (227.6 nm/px)
BBox count: 901
Opening slide: /mnt/d/microglia_data/2502 Busulfan TPO_63 TPO_Iba1_Brain - 2026-04-21 17.14.21.ndpi
  Dimensions: 159744 x 37632 px
  Resolution: 0.2276 µm/px  (227.6 nm/px)
BBox count: 1290
Opening slide: /mnt/d/microglia_data/2502 Busulfan

IndexError: index 11 is out of bounds for axis 0 with size 11

In [2]:
import numpy as np
from torch.utils.data import DataLoader
import pyvips
from pathlib import Path
import random
import matplotlib.pyplot as plt

random.seed(25)
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from skan import draw, Skeleton
import pyvips
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from scripts.filters import *
from scripts.utils import *
import os
import xml.etree.ElementTree as ET
import xml.dom.minidom
import openslide


def write_ndpa(df, NDPI_PATH, slice=1, slide=None):
    filtered_df = df[(df["brain_slice"] == slice) & (df["slide"] == slide)].copy()

    filtered_df["label"] = ""
    filtered_df["color"] = "#FF0000"
    filtered_df["width"] = filtered_df["x2"] - filtered_df["x1"]
    filtered_df["height"] = filtered_df["y2"] - filtered_df["y1"]

    filtered_df = filtered_df.rename(columns={"x1": "x", "y1": "y"})

    BOUNDING_BOXES = filtered_df[
        ["label", "color", "x", "y", "width", "height"]
    ].to_dict(orient="records")

    def px_to_ndpa(px, py, img_w, img_h, mpp_x_nm, mpp_y_nm, offset_x_nm, offset_y_nm):
        """Convert a single pixel coordinate (top-left origin) to NDPA nanometres."""
        ndpa_x = (px - img_w / 2) * mpp_x_nm + offset_x_nm
        ndpa_y = (py - img_h / 2) * mpp_y_nm + offset_y_nm
        return int(round(ndpa_x)), int(round(ndpa_y))

    def build_ndpa(boxes, img_w, img_h, mpp_x_nm, mpp_y_nm, offset_x_nm, offset_y_nm):

        fragments = []

        for i, box in enumerate(boxes):
            label = box.get("label", f"bbox_{i + 1}")
            color = box.get("color", "#FF0000")

            x, y, w, h = box["x"], box["y"], box["width"], box["height"]

            cx_px = x + w / 2
            cy_px = y + h / 2
            cx_nm, cy_nm = px_to_ndpa(
                cx_px, cy_px, img_w, img_h, mpp_x_nm, mpp_y_nm, offset_x_nm, offset_y_nm
            )

            corners_px = [
                (x, y),
                (x + w, y),
                (x + w, y + h),
                (x, y + h),
            ]
            corners_nm = [
                px_to_ndpa(
                    cx2, cy2, img_w, img_h, mpp_x_nm, mpp_y_nm, offset_x_nm, offset_y_nm
                )
                for cx2, cy2 in corners_px
            ]

            state = ET.Element("ndpviewstate", id=str(i + 1))
            ET.SubElement(state, "title").text = label
            ET.SubElement(state, "details").text = ""
            ET.SubElement(state, "x").text = str(cx_nm)
            ET.SubElement(state, "y").text = str(cy_nm)
            ET.SubElement(state, "z").text = "0"
            ET.SubElement(state, "lens").text = "20.000000"

            ann = ET.SubElement(
                state, "annotation", type="freehand", displayname=label, color=color
            )
            ET.SubElement(ann, "measuretype").text = "0"
            ET.SubElement(ann, "closed").text = "1"
            ET.SubElement(ann, "specialtype").text = "rectangle"

            pl = ET.SubElement(ann, "pointlist")
            for nx, ny in corners_nm:
                pt = ET.SubElement(pl, "point")
                ET.SubElement(pt, "x").text = str(nx)
                ET.SubElement(pt, "y").text = str(ny)

            ET.SubElement(ann, "specialtype").text = "rectangle"

            fragments.append(ET.tostring(state, encoding="unicode"))

        return fragments

    def pretty_join(fragments):
        parts = []
        for frag in fragments:
            pretty = xml.dom.minidom.parseString(frag).toprettyxml(indent="  ")
            lines = pretty.splitlines()
            body = "\n".join(l for l in lines if not l.startswith("<?xml"))
            parts.append(body.strip())

        inner = "\n".join(parts)
        return (
            '<?xml version="1.0" encoding="utf-8" standalone="yes"?>\n'
            "<annotations>\n"
            f"{inner}\n"
            "</annotations>\n"
        )

    if not os.path.isfile(NDPI_PATH):
        raise FileNotFoundError(f"NDPI file not found: {NDPI_PATH}")

    print(f"Opening slide: {NDPI_PATH}")
    slide = openslide.OpenSlide(NDPI_PATH)
    props = slide.properties

    img_w, img_h = slide.dimensions
    print(f"  Dimensions: {img_w} x {img_h} px")

    mpp_x = float(props.get(openslide.PROPERTY_NAME_MPP_X, 0))
    mpp_y = float(props.get(openslide.PROPERTY_NAME_MPP_Y, 0))

    mpp_x_nm = mpp_x * 1000
    mpp_y_nm = mpp_y * 1000
    print(f"  Resolution: {mpp_x:.4f} µm/px  ({mpp_x_nm:.1f} nm/px)")

    offset_x_nm = float(props.get("hamamatsu.XOffsetFromSlideCentre", 0))
    offset_y_nm = float(props.get("hamamatsu.YOffsetFromSlideCentre", 0))

    slide.close()

    print(f"BBox count: {len(BOUNDING_BOXES)}")
    fragments = build_ndpa(
        BOUNDING_BOXES, img_w, img_h, mpp_x_nm, mpp_y_nm, offset_x_nm, offset_y_nm
    )

    out_path = NDPI_PATH + ".ndpa"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(pretty_join(fragments))

In [3]:
df = pd.read_parquet("./results/new2/microglia_region_split.parquet")
vals = df["slide"].unique()
NDPI_DIR = "/mnt/d/microglia_data/"
ndpi_pathes = [f for f in Path(NDPI_DIR).glob("./*.ndpi")]
print(vals)
print(ndpi_pathes)

['TPO_60' 'TPO_61_EV' 'TPO_62_EV2' 'TPO_62_EV' 'TPO_63_TPO' 'TPO_64_EV2'
 'TPO_64_EV' 'TPO_65_TPO2' 'TPO_65_TPO' 'TPO_66_EV2' 'TPO_66_EV'
 'TPO_67_TPO']
[PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_60 TPO_Iba1_Brain - 2026-04-21 16.45.55.ndpi'), PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_61 EV_Iba1_Brain - 2026-04-21 16.53.13.ndpi'), PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_62 EV_2_Iba1_Brain - 2026-04-21 17.07.21.ndpi'), PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_62 EV_Iba1_Brain - 2026-04-21 17.00.49.ndpi'), PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_63 TPO_Iba1_Brain - 2026-04-21 17.14.21.ndpi'), PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_64 EV_2_Iba1_Brain - 2026-04-21 17.28.33.ndpi'), PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_64 EV_Iba1_Brain - 2026-04-21 17.21.11.ndpi'), PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_65 TPO_2_Iba1_Brain - 2026-04-21 17.43.26.ndpi'), PosixPath('/mnt/d/microglia_data/2502 Busulfan TPO_65

In [4]:
slice = 5
for i, path in enumerate(ndpi_pathes):
    write_ndpa(df, str(path), slice, vals[i])

Opening slide: /mnt/d/microglia_data/2502 Busulfan TPO_60 TPO_Iba1_Brain - 2026-04-21 16.45.55.ndpi
  Dimensions: 163840 x 39424 px
  Resolution: 0.2276 µm/px  (227.6 nm/px)
BBox count: 1958
Opening slide: /mnt/d/microglia_data/2502 Busulfan TPO_61 EV_Iba1_Brain - 2026-04-21 16.53.13.ndpi
  Dimensions: 167936 x 45824 px
  Resolution: 0.2276 µm/px  (227.6 nm/px)
BBox count: 2316
Opening slide: /mnt/d/microglia_data/2502 Busulfan TPO_62 EV_2_Iba1_Brain - 2026-04-21 17.07.21.ndpi
  Dimensions: 180224 x 44544 px
  Resolution: 0.2276 µm/px  (227.6 nm/px)
BBox count: 1810
Opening slide: /mnt/d/microglia_data/2502 Busulfan TPO_62 EV_Iba1_Brain - 2026-04-21 17.00.49.ndpi
  Dimensions: 143360 x 37120 px
  Resolution: 0.2276 µm/px  (227.6 nm/px)
BBox count: 1440
Opening slide: /mnt/d/microglia_data/2502 Busulfan TPO_63 TPO_Iba1_Brain - 2026-04-21 17.14.21.ndpi
  Dimensions: 159744 x 37632 px
  Resolution: 0.2276 µm/px  (227.6 nm/px)
BBox count: 2215
Opening slide: /mnt/d/microglia_data/2502 Busu